In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt

from model import SASRecModel


# для colab
# import sys
# sys.path.append('/content/drive/MyDrive')

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
def ndcg_at_k(rel, pred, k=10):
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    recall = 1.0 if rel in pred[:k] else 0.0
    return recall

In [ ]:
def info_nce_loss(item_emb, author_ids, temp=0.1):
    # сближает эмбеддинги книг одного автора
    batch_size = item_emb.shape[0]
    item_emb = F.normalize(item_emb, dim=1)
    sim = torch.mm(item_emb, item_emb.t()) / temp
    author_ids = author_ids.unsqueeze(0)
    mask = (author_ids == author_ids.t()).float()
    mask = mask - torch.eye(batch_size, device=mask.device)
    exp_sim = torch.exp(sim)
    pos_sum = (exp_sim * mask).sum(dim=1)
    sum_all = exp_sim.sum(dim=1) - torch.exp(torch.diag(sim))
    loss = -torch.log(pos_sum / sum_all + 1e-8)
    sum_mask = mask.sum(dim=1)
    loss = (loss * (sum_mask > 0).float()).sum() / (sum_mask > 0).float().sum() if (sum_mask > 0).any() else torch.tensor(0.0, device=loss.device)
    return loss

In [ ]:
def item_dropout(inputs, authors, categories, dropout_rate=0.15):
    # случайно заменяет 15% книг на padding
    mask = torch.rand_like(inputs.float()) > dropout_rate
    mask[inputs == 0] = True
    dropped_inputs = inputs.clone()
    dropped_inputs[~mask] = 0
    dropped_authors = authors.clone()
    dropped_authors[~mask] = 0
    dropped_categories = categories.clone()
    dropped_categories[~mask] = 0
    return dropped_inputs, dropped_authors, dropped_categories